# HU03 · Preparar los datos y definir la partición

**Proyecto:** Sistema Inteligente para clasificar y segmentar tumores cerebrales en MRI · **Sprint 1** · Fase CRISP-ML(Q): preparación de los datos

Cubre los criterios de aceptación de HU03:
1. Archivos corruptos revisados y pares imagen–máscara verificados.
2. La **prueba oficial** (1.000 imágenes) queda reservada: no se usa para entrenar ni para ajustar.
3. Riesgo de fuga en la validación documentado (BRISC no trae ID de paciente) **y mitigado agrupando imágenes muy parecidas**.
4. Preparación reproducible: semilla fija y archivos de partición guardados.
5. Cargador de datos probado con un lote.

**Idea clave:** BRISC no trae ID de paciente, así que cortes consecutivos del mismo paciente pueden caer en la partición de entrenamiento y en la de validación, inflando las métricas. Para reducir ese riesgo se calcula un *hash perceptual* de cada imagen, se forman **grupos de imágenes casi idénticas** y cada grupo completo se asigna a un solo lado de la partición.

In [ ]:
import os, re, json, glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# paleta sobria (azules, verde azulado y grises) para todos los graficos
PALETA = ["#1F3864", "#5B84B1", "#2B7A8B", "#8FB89A", "#9AA5B1"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=PALETA)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
from PIL import Image
from IPython.display import display

SEED = 42
MASK_THR = 127   # las máscaras traen valores intermedios: se consideran tumor los píxeles > 127
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ----------------------------- CONFIGURACION LOCAL -----------------------------
# Por defecto el dataset se busca en  ~/datasets  (lo deja alli descargar_brisc.py) y en  <proyecto>/data.
# Si lo tienes en otra carpeta, escribe aqui la ruta de la carpeta que contiene "classification task":
BRISC_ROOT_MANUAL = ""      # ejemplo: "D:/datasets/brisc2025"
# --------------------------------------------------------------------------------

# Carpeta de salida (figuras, resumen y particion): junto al notebook, en ./salida_sprint1
OUT = Path("salida_sprint1")
OUT.mkdir(parents=True, exist_ok=True)


def find_dirs(base, pattern, max_depth=5):
    """busca carpetas cuyo nombre coincida con el patron (sin distinguir mayusculas)"""
    base, out = Path(base), []
    if not base.exists():
        return out
    for dp, dn, fn in os.walk(base):
        depth = len(Path(dp).relative_to(base).parts)
        if depth > max_depth:
            dn[:] = []
            continue
        for d in dn:
            if re.fullmatch(pattern, d, flags=re.I):
                out.append(Path(dp) / d)
    return out


def candidatos():
    c = []
    if BRISC_ROOT_MANUAL:
        c.append(Path(BRISC_ROOT_MANUAL))
    if os.environ.get("BRISC_ROOT"):
        c.append(Path(os.environ["BRISC_ROOT"]))
    c.append(Path.home() / "datasets")          # aqui lo deja descargar_brisc.py (fuera de OneDrive)
    for p in [Path.cwd(), *Path.cwd().parents][:5]:
        c.append(p / "data")
    return c


CLS_DIRS = []
for base in candidatos():
    CLS_DIRS = find_dirs(base, r"classification[ _]task")
    if CLS_DIRS:
        break
assert CLS_DIRS, ("No encontre la carpeta 'classification task'. Ejecuta primero descargar_brisc.py "
                  "o escribe la ruta del dataset en BRISC_ROOT_MANUAL (arriba).")
CLS = CLS_DIRS[0]
ROOT = CLS.parent
SEG_DIRS = find_dirs(ROOT, r"segmentation[ _]task")
SEG = SEG_DIRS[0] if SEG_DIRS else None
print("Raiz del dataset:", ROOT)
print("Clasificacion:", CLS.name, "| Segmentacion:", SEG.name if SEG else "NO ENCONTRADA")

IMG_EXT = {".jpg", ".jpeg", ".png"}
CLASS_ALIASES = {"glioma": "glioma", "meningioma": "meningioma", "pituitary": "pituitary",
                 "no tumor": "no_tumor", "no_tumor": "no_tumor", "notumor": "no_tumor"}
CLASS_ES = {"glioma": "Glioma", "meningioma": "Meningioma", "pituitary": "Tumor de hipófisis", "no_tumor": "Sin tumor"}
CLASS_ORDER = ["glioma", "meningioma", "pituitary", "no_tumor"]
PLANE_ES = {"ax": "axial", "co": "coronal", "sa": "sagital"}


def parse_name(name):
    """brisc2025_<split>_<indice>_<codigo de tumor>_<plano>_<secuencia>.<ext> (segun el articulo)"""
    tok = re.split(r"[_ ]+", Path(name).stem)
    return (tok[3] if len(tok) > 3 else None, tok[4] if len(tok) > 4 else None)


def build_cls_table():
    rows = []
    for split_dir in sorted(CLS.iterdir()):
        if not split_dir.is_dir():
            continue
        split = split_dir.name.lower()
        for cdir in sorted(split_dir.iterdir()):
            if not cdir.is_dir():
                continue
            cls = CLASS_ALIASES.get(cdir.name.lower().replace("-", " "), cdir.name.lower())
            for p in sorted(cdir.iterdir()):
                if p.suffix.lower() in IMG_EXT:
                    code, plane = parse_name(p.name)
                    with Image.open(p) as im:
                        w, h = im.size
                    rows.append(dict(file=p.name, stem=p.stem, path=str(p.relative_to(ROOT)), split=split, cls=cls,
                                     code=code, plane=plane, width=w, height=h))
    return pd.DataFrame(rows)


def build_seg_table():
    rows = []
    if SEG is None:
        return pd.DataFrame(columns=["stem", "split", "image", "mask"])
    for split_dir in sorted(SEG.iterdir()):
        if not split_dir.is_dir():
            continue
        imgs = {p.stem: p for p in (split_dir / "images").iterdir() if p.suffix.lower() in IMG_EXT}
        msks = {p.stem: p for p in (split_dir / "masks").iterdir() if p.suffix.lower() in IMG_EXT}
        for s in sorted(set(imgs) | set(msks)):
            rows.append(dict(stem=s, split=split_dir.name.lower(),
                             image=str(imgs[s].relative_to(ROOT)) if s in imgs else None,
                             mask=str(msks[s].relative_to(ROOT)) if s in msks else None))
    return pd.DataFrame(rows)


SPLIT_FRAC = 0.20      # fracción de validación dentro del entrenamiento oficial
HASH_THRESHOLD = 16    # distancia de Hamming máxima (de 256 bits, ~6 %) para considerar dos imágenes "casi iguales"
CORR_THRESHOLD = 0.98  # correlación mínima entre miniaturas 32x32 para el control entre particiones oficiales
IMG_SIZE = 256         # tamaño de entrada (se puede bajar a 224 para transfer learning)


## 1. Tabla de imágenes y pares con máscara

In [ ]:
df_cls = build_cls_table()
df_seg = build_seg_table()
mask_by_stem = {r.stem: r["mask"] for _, r in df_seg.iterrows() if isinstance(r["mask"], str) and isinstance(r.image, str)}
df_cls["mask"] = df_cls.stem.map(mask_by_stem)
print("Imágenes:", len(df_cls), "| con máscara:", int(df_cls["mask"].notna().sum()))
print(df_cls.groupby(["split", "cls"]).size().unstack(fill_value=0))

## 2. Verificación de integridad

In [ ]:
problemas = []
n_no_bin = 0
for _, r in df_cls.iterrows():
    try:
        with Image.open(ROOT / r.path) as im:
            im.verify()
    except Exception as e:
        problemas.append(("imagen corrupta", r.path, str(e)[:60]))
    if isinstance(r["mask"], str):
        try:
            with Image.open(ROOT / r["mask"]) as m:
                m.verify()
            mk = np.asarray(Image.open(ROOT / r["mask"]).convert("L"))
            if set(np.unique(mk).tolist()) - {0, 255}:
                n_no_bin += 1
            if not (mk > MASK_THR).any():
                problemas.append(("máscara vacía", r["mask"], "sin píxeles de tumor"))
            with Image.open(ROOT / r.path) as im:
                if mk.shape != (im.size[1], im.size[0]):
                    problemas.append(("tamaño distinto imagen/máscara", r.path, f"{mk.shape} vs {im.size[::-1]}"))
        except Exception as e:
            problemas.append(("máscara corrupta", r["mask"], str(e)[:60]))
print("Problemas encontrados:", len(problemas))
print(f"Máscaras con valores intermedios (no 0/255): {n_no_bin} -> no es un error: se binarizan con umbral > {MASK_THR}")
for p in problemas[:15]:
    print("  ", p)

solo_mascara = df_seg[df_seg["mask"].notna() & df_seg.image.isna()]
solo_imagen = df_seg[df_seg.image.notna() & df_seg["mask"].isna()]
print("Máscaras sin imagen:", len(solo_mascara), "| imágenes de segmentación sin máscara:", len(solo_imagen))

## 3. Hash perceptual y grupos de imágenes casi idénticas
Se calcula un *dHash* de 256 bits (diferencias entre píxeles vecinos de una versión reducida de 17x16) y se agrupan las imágenes cuya distancia de Hamming sea ≤ umbral. El umbral no se fija a ciegas: se mide primero cuánto se parecen entre sí las imágenes del entrenamiento.

In [ ]:
def firmas(path):
    """dHash de 256 bits + miniatura 32x32 normalizada (para el control entre particiones)"""
    with Image.open(path) as im:
        g = im.convert("L")
        a = np.asarray(g.resize((17, 16), Image.LANCZOS), dtype=np.int16)
        t = np.asarray(g.resize((32, 32), Image.LANCZOS), dtype=np.float32).flatten()
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    t = (t - t.mean()) / (t.std() + 1e-6)
    return bits, t


sig = [firmas(ROOT / p) for p in df_cls.path]
BITS = np.array([s[0] for s in sig], dtype=np.float32)    # (N, 256) de 0/1
THUMB = np.array([s[1] for s in sig], dtype=np.float32)   # (N, 1024)


def hamming(A, B):
    return (A.shape[1] - (A @ B.T + (1 - A) @ (1 - B).T)).astype(np.int32)


def agrupar(idx, thr):
    """union-find: une imágenes con distancia de Hamming <= thr (idx = posiciones en BITS)"""
    D = hamming(BITS[idx], BITS[idx])
    n = len(idx)
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    ii, jj = np.nonzero(np.triu(D <= thr, 1))
    for i, j in zip(ii, jj):
        ra, rb = find(int(i)), find(int(j))
        if ra != rb:
            parent[rb] = ra
    return np.array([find(i) for i in range(n)]), D


train_pos = np.nonzero((df_cls.split == "train").to_numpy())[0]
test_pos = np.nonzero((df_cls.split == "test").to_numpy())[0]
train_off = df_cls.iloc[train_pos].reset_index(drop=True)
test_off = df_cls.iloc[test_pos].reset_index(drop=True)

# diagnóstico: distancia al vecino más cercano dentro del entrenamiento
g, D0 = agrupar(train_pos, HASH_THRESHOLD)
np.fill_diagonal(D0, 999)
vecino = D0.min(axis=1)
print("Distancia de Hamming (de 256) al vecino más cercano dentro del entrenamiento, percentiles 5/25/50/75/95:",
      [int(np.percentile(vecino, q)) for q in (5, 25, 50, 75, 95)])
print("\nUmbral -> grupos | tamaño del mayor | grupos con más de una imagen")
for thr in (8, 16, 24, 32):
    gg, _ = agrupar(train_pos, thr)
    sz = pd.Series(gg).value_counts()
    print(f"  {thr:>3} -> {len(sz):>5} | {int(sz.iloc[0]):>4} | {int((sz > 1).sum()):>4}")

train_off["group"] = g
sizes = train_off.groupby("group").size()
print(f"\nUmbral elegido: {HASH_THRESHOLD}")
print("Imágenes de entrenamiento oficial:", len(train_off), "| grupos:", len(sizes))
print("Grupos con más de una imagen:", int((sizes > 1).sum()), "| tamaño del mayor grupo:", int(sizes.max()))
if sizes.max() > 0.02 * len(train_off):
    print("ATENCIÓN: hay un grupo muy grande; baja HASH_THRESHOLD para no mezclar imágenes distintas.")

# informativo: imágenes de la prueba oficial casi idénticas a alguna de entrenamiento (fuga entre las particiones OFICIALES)
corr = (THUMB[test_pos] @ THUMB[train_pos].T) / THUMB.shape[1]
dh = hamming(BITS[test_pos], BITS[train_pos])
print("\nImágenes de la prueba oficial (de", len(test_off), ") con un vecino casi idéntico en entrenamiento:")
print(f"  por dHash (distancia <= {HASH_THRESHOLD}):", int((dh.min(axis=1) <= HASH_THRESHOLD).sum()))
print(f"  por correlación de miniaturas (>= {CORR_THRESHOLD}):", int((corr.max(axis=1) >= CORR_THRESHOLD).sum()))
test_off["vecino_cercano_train"] = corr.max(axis=1) >= CORR_THRESHOLD

## 4. Partición entrenamiento / validación
Estratificada por **clase y plano** y respetando los grupos (ningún grupo queda repartido entre entrenamiento y validación). La prueba oficial no se toca.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

estrato = train_off.cls.astype(str) + "_" + train_off.plane.astype(str)
n_splits = int(round(1 / SPLIT_FRAC))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf.split(train_off, estrato, groups=train_off.group))

train_off["particion"] = "train"
train_off.loc[val_idx, "particion"] = "val"
test_off["particion"] = "test"
final = pd.concat([train_off, test_off], ignore_index=True)

# comprobaciones
g_train = set(final[final.particion == "train"].group.dropna())
g_val = set(final[final.particion == "val"].group.dropna())
assert not (g_train & g_val), "hay grupos repartidos entre entrenamiento y validación"
assert set(final[final.particion == "test"].file) == set(df_cls[df_cls.split == "test"].file), "la prueba oficial cambió"
print("Entrenamiento:", (final.particion == "train").sum(), "| validación:", (final.particion == "val").sum(), "| prueba oficial:", (final.particion == "test").sum())
print("\nImágenes por clase y partición:")
display(pd.crosstab(final.cls, final.particion).reindex(CLASS_ORDER))
print("Imágenes por plano y partición:")
display(pd.crosstab(final.plane, final.particion))
vv = final[final.particion != "test"]
print("Proporción de validación por clase:", vv.groupby("cls").particion.apply(lambda s: (s == "val").mean()).round(3).to_dict())
print("Proporción de validación por plano:", vv.groupby("plane").particion.apply(lambda s: (s == "val").mean()).round(3).to_dict())

## 5. Guardar la partición (reproducible)

In [ ]:
part_dir = OUT / "particion"
part_dir.mkdir(exist_ok=True)
cols = ["file", "path", "cls", "plane", "particion", "group", "mask"]
for nombre, sub in final.groupby("particion"):
    sub[cols].to_csv(part_dir / f"particion_{nombre}.csv", index=False)
final[cols].to_csv(part_dir / "particion_completa.csv", index=False)

config = {"semilla": SEED, "fraccion_validacion": SPLIT_FRAC, "umbral_hash_256bits": HASH_THRESHOLD, "umbral_binarizar_mascara": MASK_THR,
          "tamano_imagen": IMG_SIZE, "relleno_a_cuadrado": True,
          "prueba_con_vecino_casi_identico_en_train": int(test_off["vecino_cercano_train"].sum()),
          "normalizacion": {"media": [0.485, 0.456, 0.406], "desvio": [0.229, 0.224, 0.225]},
          "conteos": final.particion.value_counts().to_dict(),
          "grupos_entrenamiento_oficial": int(train_off.group.nunique())}
(part_dir / "config_preparacion.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Archivos guardados en", part_dir, ":", sorted(p.name for p in part_dir.iterdir()))

## 6. Cargador de datos
Devuelve imagen normalizada (C×H×W), clase, máscara y un indicador `tiene_mascara`. Las imágenes **sin tumor** devuelven una máscara vacía con `tiene_mascara = False` (así se puede ignorar su pérdida de segmentación o tratarla como máscara vacía en un modelo multitarea).

Decisiones de preprocesamiento (HU03): las imágenes tienen tamaños variables, así que primero se **rellenan a cuadrado con ceros (sin deformar)** y luego se redimensionan a `IMG_SIZE`; interpolación bilineal para imágenes y *nearest* para máscaras; las máscaras se binarizan con umbral > 127; normalización con media y desvío de ImageNet (para transfer learning). El aumento de datos se define y ajusta en el Sprint 2 (HU08).

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class BriscDataset:
    def __init__(self, tabla, root=ROOT, img_size=IMG_SIZE):
        self.t = tabla.reset_index(drop=True)
        self.root, self.s = Path(root), img_size

    def __len__(self):
        return len(self.t)

    @staticmethod
    def _cuadrar(im, modo):
        """rellena con ceros hasta un cuadrado (sin deformar), centrando la imagen"""
        w, h = im.size
        if w == h:
            return im
        lado = max(w, h)
        lienzo = Image.new(modo, (lado, lado), 0)
        lienzo.paste(im, ((lado - w) // 2, (lado - h) // 2))
        return lienzo

    def __getitem__(self, i):
        r = self.t.iloc[i]
        img = self._cuadrar(Image.open(self.root / r.path).convert("RGB"), "RGB").resize((self.s, self.s), Image.BILINEAR)
        x = ((np.asarray(img, dtype=np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)
        y = CLASS_ORDER.index(r.cls)
        tiene = isinstance(r["mask"], str)
        if tiene:
            m = self._cuadrar(Image.open(self.root / r["mask"]).convert("L"), "L").resize((self.s, self.s), Image.NEAREST)
            mk = (np.asarray(m) > MASK_THR).astype(np.float32)[None]
        else:
            mk = np.zeros((1, self.s, self.s), dtype=np.float32)
        return x, y, mk, tiene


ds_train = BriscDataset(final[final.particion == "train"])
ds_val = BriscDataset(final[final.particion == "val"])
ds_test = BriscDataset(final[final.particion == "test"])
print("Tamaños -> entrenamiento:", len(ds_train), "| validación:", len(ds_val), "| prueba oficial:", len(ds_test))

### Prueba con un lote

In [ ]:
try:
    import torch
    from torch.utils.data import DataLoader
    loader = DataLoader(ds_train, batch_size=8, shuffle=True, num_workers=0)
    xb, yb, mb, tb = next(iter(loader))
    print("PyTorch OK ->", tuple(xb.shape), tuple(yb.shape), tuple(mb.shape), tb.dtype)
except ImportError:
    items = [ds_train[i] for i in range(8)]
    xb = np.stack([it[0] for it in items]); yb = np.array([it[1] for it in items])
    mb = np.stack([it[2] for it in items]); tb = np.array([it[3] for it in items])
    print("NumPy OK ->", xb.shape, yb.shape, mb.shape)

assert tuple(xb.shape)[1:] == (3, IMG_SIZE, IMG_SIZE), "forma de imagen inesperada"
assert tuple(mb.shape)[1:] == (1, IMG_SIZE, IMG_SIZE), "forma de máscara inesperada"
print("Clases del lote:", [CLASS_ORDER[int(i)] for i in yb])
print("Media/desvío de la imagen normalizada:", float(xb.mean()), float(xb.std()))

fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for k, ax in enumerate(axes.flat):
    img = np.asarray(xb[k]).transpose(1, 2, 0) * STD + MEAN
    ax.imshow(np.clip(img, 0, 1))
    m = np.asarray(mb[k][0])
    if m.sum() > 0:
        ov = np.zeros((*m.shape, 4)); ov[..., 0] = 1; ov[..., 3] = 0.45 * m
        ax.imshow(ov)
    ax.set_title(CLASS_ES[CLASS_ORDER[int(yb[k])]], fontsize=9); ax.axis("off")
plt.tight_layout(); fig.savefig(OUT / "lote_de_prueba.png", dpi=120); plt.show()

## Resultado de HU03
- Partición guardada en `particion/` (entrenamiento, validación y prueba oficial) con semilla fija y grupos de imágenes parecidas.
- Riesgo residual: sin ID de paciente, la agrupación por hash **reduce pero no elimina** la fuga (y en la prueba oficial hay imágenes muy parecidas a las de entrenamiento: ver el conteo de la sección 3). La **prueba oficial** sigue siendo la medida confiable y es la única que se usa para reportar el resultado final.
- Siguiente paso (Sprint 2): arquitectura y modelo base (HU05) usando este cargador.